# nextup-lab quickstart

Drop your experiments in, get the next batch to run. Chemistry, chemical engineering, biology — whatever you're optimizing.

1. Put your tested experiments + candidate conditions in a CSV.
2. Run all cells.
3. The last cell prints the next experiments to run.

_CPU only. Bigger pools take longer (linear in candidates, cubic in tested rows)._

## 1. Install

In [ ]:
!pip install -q nextup-lab

## 2. Upload your CSV

**Format:**
- One row per experiment (tested OR candidate).
- One column for what you're measuring (yield, selectivity, EC50…). Tested rows have the value; candidates leave it blank.
- Other columns describe the conditions: numbers (temperature, equivalents), categories (catalyst, solvent), or SMILES strings (substrates, ligands).

Mix and match — auto-detected.

**No CSV yet?** Skip this cell; the next one loads a built-in demo.

In [ ]:
import pandas as pd

try:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(iter(uploaded))
    df = pd.read_csv(csv_name)
except Exception:
    # Local / no upload — build the demo
    import numpy as np
    rng = np.random.default_rng(0)
    n = 40
    df = pd.DataFrame({
        'temp_C':    rng.uniform(20, 120, n),
        'catalyst':  rng.choice(['Pd-A', 'Pd-B', 'Ni-A', 'Ni-B', 'Cu-A'], n),
        'solvent':   rng.choice(['toluene', 'THF', 'DMF'], n),
    })
    # First 8 rows: tested
    df['yield'] = [None] * n
    df.loc[:7, 'yield'] = [22, 8, 41, 15, 53, 33, 19, 28]
    print('Loaded demo dataset (40 conditions, first 8 tested).')

df.head(10)

## 3. Pick the column to maximize + how many experiments to run next

In [ ]:
OBJECTIVE = 'yield'    # the column you want to maximize
K         = 4         # how many experiments to suggest next

history    = df[df[OBJECTIVE].notna()].reset_index(drop=True)
candidates = df[df[OBJECTIVE].isna()].drop(columns=[OBJECTIVE]).reset_index(drop=True)
print(f'Tested: {len(history)}   Candidates: {len(candidates)}')

## 4. Get the next batch

Output:
- `nextup_rank` — 1 = run this one first
- `nextup_score` — model's confidence the row is in the top-5% region (higher = more confident)

In [ ]:
from nextup_lab import suggest

next_batch = suggest(history, candidates, objective=OBJECTIVE, k=K)
next_batch

## 5. Run those experiments → come back and fill in the values

Open your CSV, fill in `yield` (or whatever you named it) for the rows above, save, and re-run from cell 2. The next suggestions improve as your data grows.

---

**Found a bug or have a request?** Open an issue at the repo. Stars help. ⭐